# Pruning Pipeline

---
### Sensitivity Analysis
LD-Pruner Sensitivity Analysis for SD-Turbo UNet pruning ratio arrangement.

#### 1. Channel Selection

- For each (block, component, ratio) candidate, install magnitude-based channel masks on the student UNet, run the student at the same inputs, call scheduler.step to obtain the student's post-step latent, then remove the masks.

#### 2. LD-score

- **Teacher Data Collection:** For each calibration prompt, run the teacher (unpruned) UNet through a full SD-Turbo 4-step inference (timesteps fixed to `[999, 749, 499, 249]`). At every step, we capture the pre-step latent as the start. 

- **Student Inference:** Call `scheduler.step` to obtain the student's post-step latent, then remove the masks. And compare the `post-step latent` between student and teacher.

- LD score is computed per-timestep (to avoid high-noise timesteps dominating the std term) and summed across the 4 timesteps:
  - `avg_dist_t = ||mean_teacher_t - mean_student_t||`
  - `std_dist_t = ||std_teacher_t  - std_student_t||`
  - `ld_t       = avg_dist_t + std_dist_t`
  - `ld_score   = sum_t(ld_t)`

#### 3. Pruning Ratio Arrangement

Converts LD scores into a concrete per-block pruning config via binary search on a sensitivity threshold.

- Conv / FFN blocks (threshold-based):
    - Protected blocks (`down_blocks.0`, `up_blocks.3`) are **never pruned** for resnet/ffn — shallow input/output layers
    - For each `(block, comp)`, we have a sensitivity curve: `ratio → ld_score`
    - Binary search for a global threshold `T` such that the total parameter removal matches the target. 
    - For each component, find the pruning ratio where its curve crosses `T` via linear interpolation, then snap to the nearest granularity (e.g. 0.05):
        - raw_ratio = interpolate(curve, threshold=T)
        - snapped_ratio   = round(raw_ratio / snap) * snap

**Attention layers (hardcoded, per-head):**

- Attention pruning is **discrete**: 8 heads total, each removal is 1/8 = 12.5%
- Heads-to-remove per block are set manually based on sensitivity analysis:

    | Block | self_attn | cross_attn |
    |---|---|---|
    | mid_block | 2 heads (25%) | 2 heads (25%) |
    | down_blocks.2 | 1 head (12.5%) | 2 heads (25%) |
    | up_blocks.1 | 1 head (12.5%) | 2 heads (25%) |
    | down_blocks.0/1, up_blocks.2/3 | - | 1 head (12.5%) |

- Output: A `pruning_config_XX pct_auto.json` file with per-block ratios, passed directly to the Taylor Softmask pipeline.

#### 4. Work Flow

4.1 Load Model

4.2 Calibration data(Teacher Model)

- Run teacher `4-step inference` per prompt and collect per-timestep samples. We need the actual semantic denoising trajectory.
- This is creating a standard example for the comparison. The difference between this and pruned model is caused by pruning. 
- `noisy_latents`, `encoder_hidden_states`, `teacher_pred`, `teacher_next_latent` are saved in calibration data. 

4.3 Match Components and Blocks by Name

4.4 Compute LD Scores for All Combinations

- **Bucket for each timestep:** Latent magnitude varies from different timesteps, so we `separate timesteps into buckets` and calculate LD score independently. 

- **Reuse of teacher latents:** to avoid unnecessary compute. Use precomputed teacher latents, no forward pass here, just read from calibration data: `teacher_next_latent`

- **Student Model Computing:**
    - For each `(block, comp, ratio)` pack, `install_masks(components, block, comp, ratio)`
    - Run a `Single-step inference` with the same input as teacher model: Error will not accumulate, each timestep is independent. And easier to identify the problem. 
    - Compare the difference between student's latent and teacher's latent
    - Use LD Score to quantify the difference:
        - `avg_dist = torch.norm(teach.mean(dim=0) - s_stack.mean(dim=0)).item()`
        - `std_dist = torch.norm(teach.std(dim=0) - s_stack.std(dim=0)).item()`
        - `total_avg += avg_dist`
        - `total_std += std_dist`
        - `ld_score = total_avg + total_std`

4.5 Plot and Save Report

---
### Pruning



#### 1. Taylor Score

Identifies **which channels** are least important and can be safely removed.

- For each `(block, comp)`, compute per-channel importance:
  - `Grad` comes from: `loss = MSE(student_output, teacher_pred)`, then `loss.backward()`
  - `teacher_pred` is read directly from `calib_data`
  - `score[c] = |weight_c · grad_c|.mean()`

- Computed **per-timestep** then combined via `avg` or `max` across timesteps (controlled by `TAYLOR_MODE`)

#### 2. Soft Masking

Applies channel masks during gradient training so the model can adapt before physical pruning.

- Build soft masks from Taylor scores: low-score channels are masked to zero gradient

- Run gradient steps on masked student, cycling through `calib_data` with `MSE(student_output, teacher_pred)

- Re-Evaluation Step: periodically rebuild masks (`reeval_interval`): channels can revive or be newly masked based on updated score. 



#### 3. Work Flow

3.1 Param Parser and Load Pruning Configs

- Set up all configs and params, call `sp_core.taylor_softmask_pipeline()`

3.2 Load Calibration Data

- Reuse the same `calib_data` generated in Sensitivity Analysis
- Each sample already contains `noisy_latents`, `encoder_hidden_states`, `teacher_pred`, `teacher_next_latent`

3.3 Taylor Score Channel Selection + Softmask

- Channel Selection: 
    - Warm-Up: accumulate Taylor importance scores via forward/backward for the first-time soft-masking. 

    - Taylor Importance: 
        - Bucket `calib_data` by timestep
        - Run student forward → `MSE(student, teacher_pred)` → backward → `Gradients`
        - `score = (weight * grad).abs().mean(dim=tuple(range(1, weight.dim())))` Taylor score is approximately `|w · ∂L/∂w|`, both weight and contribution of a channel matters. 
        - Combine across timesteps via avg or max → final `score_accum`
            - Timesteps Average Mode: taylor score is the mean score of this channel in all timesteps. 
            - Timesteps Maximum Mode: taylor score is the maximum score of this channel in all timesteps. 

- Soft Masking: 
    - Rampup: Keep loss more stable during the start of soft-masking process. 
        - Initially have a low masking ratio
        - Masking ratio grows linearly when rebuilding mask in `reeval_interval` step. Until reaching the target masking ratio. 
    - Build initial soft masks from `score_accum`
    - Cycle through Taylor Importance evaluation step
    - Re-Evaluation Phase: periodically rebuild masks with rampup until target pruning ratio is reached. Masked channels can revive in this phase. 

- Physical pruning: 
    - Use converged Taylor scores to permanently remove low-importance channels (DG pruning)
    - Optional attention head pruning per block config
    - Save pruned checkpoint

---
### Distillation



### 1. Multi-Phase Step-wise Knowledge Distillation

**What this is:** Step-wise teacher-student matching distillation, NOT standard Consistency Distillation.

| | Standard Consistency Distillation (CM) | Our approach |
|---|---|---|
| Target | `f(x_t, t) → x_0` in one step from any timestep | Match teacher's post-step latent at each denoising step |
| Loss | Self-consistency between adjacent timesteps | `MSE(student_next, teacher_next)` per step |
| Student output | Can skip directly to x_0 | Still a standard diffusion model |

**Training design:**
- Fixed 4-step inference (`n_steps_min = n_steps_max = 4`), matching SD-Turbo's native schedule `[999, 749, 499, 249]`. Fixed steps give a stable, consistent gradient signal.
- SD-Turbo native init: `pure Gaussian × init_noise_sigma`
- **Shared input:** Every step, student and teacher both receive `t_latent_in` (teacher's latent before its update). Student's output is never fed back — teacher's trajectory drives the next input.
- **Detach:** `t_latent_in = t_latent.detach().clone()` — each step's gradient is isolated to that step's UNet forward only.

**Multi-Phase (both at native 512px):**
- Phase 1 — Main: primary quality recovery (`main_steps`, default 45000)
- Phase 2 — Finetune: low LR final polish (`finetune_steps`, default 5000, skippable)

**Learning Rate (cosine, continuous across both phases):**
- `lr_max = 2e-5` → `lr_min = 1e-6`, warmup 200 steps

### 2. Loss Function

**Prediction-space loss (per denoising step):**

```
raw_loss = MSE(s_pred, t_pred) + λ_l1 × L1(s_pred, t_pred)
```

- `s_pred`, `t_pred` — UNet epsilon predictions (not latents). Comparing predictions directly avoids EulerAncestral's stochastic noise: `scheduler.step()` samples fresh random ε each call, so teacher and student latents after stepping diverge stochastically even from the same input. Prediction-space loss bypasses the scheduler entirely.
- `L1` alongside `MSE`: MSE minimises squared error and tends to average valid outputs → blur. L1 penalises all deviations equally → better high-frequency detail preservation. Default `λ_l1 = 0.1`.

**Per-timestep EMA normalisation:**

```
ema_loss[t] = 0.99 × ema_loss[t] + 0.01 × raw_loss
norm_loss   = raw_loss / (ema_loss[t] + 1e-8)
```

t=999 (high noise) produces raw MSE ~100× larger than t=249 (near-clean). Without normalisation, high-noise timesteps dominate gradients and the model ignores near-clean steps. EMA normalisation makes each timestep contribute equally.

**Attention map loss (optional, requires `lambda_attn > 0`):**

```
attn_loss = mean over hooked layers( MSE(s_attn_map, t_attn_map.detach()) )
```

- Matches head-averaged attention weights `[B, seq_q, seq_k]` between student and teacher at selected intermediate layers.
- Shape is head-count agnostic — works even though student has fewer channels than teacher after pruning.
- Same per-timestep EMA normalisation applied with separate `ema_attn[t]` dict.
- Linear warmup: `λ_now = λ_attn × min(1, step / attn_warmup_steps)` — prevents attention loss from interfering before prediction quality stabilises.

**Final combined loss:**

```
total_loss = Σ_t(norm_pred_loss_t + λ_attn_now × norm_attn_loss_t)
loss = total_loss / n_steps / grad_accum_steps
```

`grad_accum_steps` scaling keeps gradient magnitude consistent regardless of accumulation window size.

### 3. Work Flow

**Two modes: Live mode (default) vs Cache mode**

| | Live mode | Cache mode |
|---|---|---|
| Teacher on GPU | Yes (~3–5 GB) | No (offloaded to CPU) |
| Teacher runs every step | Yes | No — reads `.pt` files |
| Batch size | 1 (tight VRAM) | 2+ (freed VRAM) |
| Setup required | None | Run `gen_teacher_cache.py` first |

---

**3.1 Load models (student and teacher, separate schedulers)**

Two independent `EulerAncestralDiscreteScheduler` instances: `pipe.scheduler` (student) and `teacher_scheduler` (teacher). Both set to the same timesteps each iteration. Separate schedulers prevent stateful variables (`prev_derivative`, `dt`) from being overwritten when teacher and student calls are interleaved.

**3.2 (Cache mode only) Pre-compute teacher cache**

```bash
python prune/gen_teacher_cache.py \
    --run-config prune/sp_distill_config.yaml \
    --cache-dir teacher_cache --n-seeds 5
```

Saves per `(prompt_idx, seed)`: `cache_dir/{p:05d}_{s:04d}/step_{i}.pt` containing `input_latent`, `t_pred`, `next_latent`, `attn_maps`. ~5 GB total. Skips already-completed trajectories (safe to resume).

**3.3 Multi-step Distillation loop**

- Randomly choose `1–4` inference steps (fixed at 4 by default: `n_steps_min = n_steps_max = 4`)
- SD-Turbo native init: `pure Gaussian × init_noise_sigma`
- **Per denoising step:**
  - Save `t_latent_in = t_latent.detach().clone()` — shared input for both teacher and student
  - *Live mode:* Teacher forward (fp16, no grad) → `t_pred`; teacher advances `t_latent`
  - *Cache mode:* Load `t_latent_in`, `t_pred`, `t_latent` from `.pt` file — no teacher GPU call
  - Student forward (fp32, grad) from `t_latent_in` → `s_pred`
  - Compute prediction loss + attention map loss (if enabled)
  - Accumulate `total_loss`

**3.4 Optimizer and Backward Propagation**

- Gradient accumulation: `zero_grad` only at start of window; loss scaled by `1/grad_accum_steps`
- `loss.backward()` — gradient flows only through current step's student forward
- `clip_grad_norm_` — rescales if L2 norm exceeds `grad_clip` (default 1.0)
- `optimizer.step()` + `lr_scheduler.step()` (cosine decay)
- EMA update: `ema_params = 0.999 × ema_params + 0.001 × live_params` (on CPU)

**3.5 Logging and Save Checkpoint**

- Log every `log_every` steps: `loss=` (normalised), `raw=` (absolute prediction MSE+L1), `attn=` (if enabled)
- Save EMA weights every `save_every` steps; keep last `keep_checkpoints`
- Phase-end checkpoint: `distill_{phase}_done.safetensors`
- Final: `distill_final.safetensors`

---

# To-do in Future



---

# Change Log



### Attention Map Distillation Implementation

**New files:**
- `prune/attn_map_capture.py` — `AttentionMapRegistry` + `CaptureAttnProcessor`
- `prune/gen_teacher_cache.py` — offline teacher trajectory pre-computation

**Modified files:**
- `prune/sp_distill.py` — cache mode, `CachedTeacherDataset`, attn loss integration
- `prune/sp_distill_config.yaml` — new attn distillation config block

---

#### `prune/attn_map_capture.py` (new)

Replaces diffusers' default `AttnProcessor2_0` (which uses fused CUDA `scaled_dot_product_attention` that does not expose intermediate weights) with a custom `CaptureAttnProcessor` that computes attention weights manually and stores the head-averaged result.

**`CaptureAttnProcessor`**: faithful re-implementation of `AttnProcessor`. Handles self-attn and cross-attn, `norm_cross`, `group_norm`, 4-D channel-first input, `residual_connection`, and `rescale_output_factor`. Stores `[B, seq_q, seq_k]` float32 detached tensor in a shared dict after each forward.

**`AttentionMapRegistry`**: lifecycle manager. `register(unet, layer_keys)` installs processors; `clear()` drops captured maps between forward passes; `restore(unet)` puts back original processors at end of training. Navigates module tree by dot-path via `_get_module` (supports integer indices for `ModuleList`).

---

#### `prune/gen_teacher_cache.py` (new)

Pre-computes the teacher trajectory for every `(prompt_idx, seed)` pair before training starts, so the teacher UNet does not need to reside on GPU during distillation.

**Per-trajectory files** (`cache_dir/{p:05d}_{s:04d}/step_{i}.pt`):
```
input_latent  [1, 4, H/8, W/8]  fp16   UNet input at this denoising step
t_pred        [1, 4, H/8, W/8]  fp16   teacher epsilon prediction
next_latent   [1, 4, H/8, W/8]  fp16   teacher latent after scheduler.step
attn_maps     {layer_key: [1, seq_q, seq_k] fp16}   head-averaged
```

Noise is seeded deterministically: `torch.manual_seed(seed * 10000 + p_idx)` — ensures exact reproducibility across interrupted/resumed runs. Skips directories where all step files already exist.

**Usage:**
```bash
python prune/gen_teacher_cache.py --run-config prune/sp_distill_config.yaml \
    --cache-dir teacher_cache --n-seeds 5
```

---

#### `prune/sp_distill.py` (modified)

**New class `CachedTeacherDataset`**: scans `cache_dir` for fully-populated trajectory directories, returns `(prompt_text, prompt_idx, seed)` triples. Only trajectories where all `step_0..step_{n-1}.pt` exist are included.

**New CLI / YAML args:**

| Arg | Default | Purpose |
|---|---|---|
| `--teacher-cache-dir` | null | Enable cache mode (offloads teacher from GPU) |
| `--lambda-attn` | 0.0 | Attention map loss weight |
| `--attn-warmup-steps` | 500 | Steps to ramp `lambda_attn` 0 → target |
| `--attn-layers` | [] | Dot-paths of modules to hook |

**Training loop changes:**

- `use_cache = bool(teacher_cache_dir)`: when True, teacher UNet is immediately moved to CPU after pipeline load, freeing ~3–5 GB VRAM.
- Phase dataset switches to `CachedTeacherDataset` in cache mode.
- Batch unpacking handles both `str` (live mode) and `(str, int, int)` (cache mode) batches.
- Per-step loop: if `use_cache`, loads `t_latent_in / t_pred / t_latent / t_attn_maps` from `.pt` files; otherwise runs teacher forward + optionally captures via `teacher_registry`.
- `student_registry.clear()` called before each student forward to reset captured maps.
- Attention loss: per-layer MSE averaged over matched layers, EMA-normalised per timestep (same scheme as prediction loss), weighted by `lambda_attn * min(1, step / attn_warmup_steps)`.
- `total_attn_loss` added to `total_loss` before the grad-accum scaling step.
- Log line extended: `attn={raw_attn_val:.5f}` appended when attn distillation is active.
- At end of training: `student_registry.restore()` and `teacher_registry.restore()` clean up all installed processors.

---

#### `prune/sp_distill_config.yaml` (modified)

Added section at end of file:
```yaml
teacher_cache_dir:  null
lambda_attn:        0.02
attn_warmup_steps:  500
attn_layers:
  - mid_block.attentions.0.transformer_blocks.0.attn1
  - mid_block.attentions.0.transformer_blocks.0.attn2
  - down_blocks.2.attentions.0.transformer_blocks.0.attn1
  - down_blocks.2.attentions.0.transformer_blocks.0.attn2
  - up_blocks.0.attentions.0.transformer_blocks.0.attn1
  - up_blocks.0.attentions.0.transformer_blocks.0.attn2
  - up_blocks.1.attentions.0.transformer_blocks.0.attn1
  - up_blocks.1.attentions.0.transformer_blocks.0.attn2
```

---

#### How to run

```bash
# Step 1 — pre-compute teacher cache (run once, ~5 GB output)
python prune/gen_teacher_cache.py \
    --run-config prune/sp_distill_config.yaml \
    --cache-dir teacher_cache --n-seeds 5

# Step 2 — distillation with attn map loss + teacher off GPU
python prune/sp_distill.py \
    --teacher-cache-dir teacher_cache \
    --lambda-attn 0.02 \
    --batch-size 2          # extra VRAM from no teacher allows larger batch
```

To run **without** the cache (teacher stays on GPU, attn maps captured live):
```bash
python prune/sp_distill.py --lambda-attn 0.02
# teacher_cache_dir remains null → live teacher mode
```

### Attention Map Distillation

**Motivation:** Current distillation only supervises the UNet's final output (MSE + L1). For pruned intermediate layers, the gradient signal travels through a long chain — transformer blocks deep in the network receive sparse, noisy learning signal. Adding attention map distillation creates direct supervision shortcuts to intermediate layers without imposing tight feature-value constraints (avoids capacity-gap overfitting where the smaller student can't faithfully replicate the teacher's exact representations).

**Why attention maps over feature distillation:**
- After channel pruning, teacher and student have different channel dimensions → feature value matching requires projection adapters and creates a tight constraint that may harm student's own expression
- Head-averaged attention maps (`attn.mean(dim=1)`) have shape `[B, seq_q, seq_k]` — **head-count agnostic**, completely bypasses the dimension mismatch problem
- Captures structural information (what attends to what) rather than raw feature values

#### 1. Zone-Aware Layer Selection

| Zone | Pruning | Strategy |
|---|---|---|
| Shallow (down 0/1, up 2/3) | 0% | Skip — no pruning damage to repair |
| Mid (down 2, up 1) | 15–20% | Attention map distillation |
| Deep (mid_block, down 3, up 0) | 35–45% | Attention map distillation (highest priority) |

**Default hook layers** (self-attn + cross-attn per block):
- `mid_block.attentions.0.transformer_blocks.0.attn1/attn2`
- `down_blocks.2.attentions.0.transformer_blocks.0.attn1/attn2`
- `up_blocks.0.attentions.0.transformer_blocks.0.attn1/attn2`
- `up_blocks.1.attentions.0.transformer_blocks.0.attn1/attn2`

#### 2. Hook Implementation

Standard diffusers `AttnProcessor2_0` uses `F.scaled_dot_product_attention` which does not expose intermediate attention weights. We replace the processor on target modules with a custom `CaptureAttnProcessor` that:
1. Computes Q, K, V projections manually
2. Computes `attn_weights = softmax(Q @ K^T * scale)`  →  `[B*heads, seq_q, seq_k]`
3. Stores head-averaged map: `.view(B, heads, seq_q, seq_k).mean(dim=1)`  →  `[B, seq_q, seq_k]`
4. Completes the attention output normally: `attn_weights @ V → to_out`

An `AttentionMapRegistry` manages registration/restore/clear lifecycle.

#### 3. Loss

```
loss_attn = mean over layers( MSE(s_map, t_map.detach()) )
```

- Same per-timestep EMA normalization as prediction loss (prevents high-noise timesteps dominating)
- Combined: `total_loss = norm_pred_loss + λ_attn * norm_attn_loss`
- Default `λ_attn = 0.05` (tunable)

#### 4. Teacher Output Pre-computation (VRAM Optimization)

Running teacher alongside student consumes ~1.7 GB (fp16 weights) + ~2–3 GB (forward activations) on GPU. Pre-computing teacher outputs offline eliminates the teacher from GPU during training entirely, freeing ~3–5 GB VRAM to increase batch size.

**`gen_teacher_cache.py` script:**
- For each `(prompt_idx, seed)` pair, run teacher 4-step denoising with fixed `torch.manual_seed(seed)`
- Save per step: `{input_latent, t_pred, next_latent, attn_maps: {layer_key: [B, seq_q, seq_k]}}`
- Format: `cache_dir/{prompt_idx:05d}_{seed:04d}/step_{i}.pt`
- Storage estimate: ~1 MB per trajectory × 5 seeds × 1000 prompts ≈ **5 GB**

**Training loop change:**  When `teacher_cache_dir` is set, skip teacher UNet load entirely; load cached tensors per batch instead.

#### 5. New Files / Modified Files

| File | Change |
|---|---|
| `prune/attn_map_capture.py` | **New** — `AttentionMapRegistry` + `CaptureAttnProcessor` |
| `prune/gen_teacher_cache.py` | **New** — offline teacher trajectory cache generation |
| `prune/sp_distill.py` | Add `--teacher-cache-dir`, `--lambda-attn`, `--attn-layers`; integrate cache loading + attn loss |
| `prune/sp_distill_config.yaml` | Add `teacher_cache_dir`, `lambda_attn`, `attn_layers` keys |

### 2:4 Sparse Computing

**Context:** NVIDIA Ampere Sparse Tensor Cores support 2:4 structured sparsity — exactly 2 of every 4 consecutive weights in each row are non-zero. The hardware delivers ~**2× GEMM throughput** and ~**1.75× weight memory reduction** using a compressed representation (50% values + small index metadata).

Target device (Orange Pi ARM) lacks Sparse Tensor Cores, but the workflow serves as a sponsor-facing demonstration and provides real speedup on Ampere-class GPUs used for the showcase.

---

#### 1. Combined Compression Pipeline

```
Phase 1: Channel Pruning              858M → 647M  (-26%, structural)
Phase 2: Distillation (+ attn maps)   recover quality after channel pruning
Phase 3: 2:4 Sparsification           apply 2:4 magnitude mask to attention Linear layers
Phase 4: 2:4 Short Distillation       5k–10k steps, frozen mask, STE backward
Inference: SparseSemiStructuredTensor ~2× GEMM speedup on Ampere
```

---

#### 2. How 2:4 Sparse Computation Works

**Storage format:**

Every 4 consecutive values in each row must have exactly 2 non-zeros. The compressed representation stores:
- **Values tensor**: `[out, in/2]` — only the non-zero values (half the weight matrix)
- **Metadata tensor**: `[out, in/16]` — 2-bit indices indicating which 2 positions in each group of 4 are non-zero

```
Dense row:      [w0, 0, w2, 0,   w4, w5,  0, 0,  ...]
                  ↑group 0↑       ↑group 1↑
Compressed:     values  = [w0, w2, w4, w5, ...]
                metadata= [(0,2),  (0,1),  ...]   ← 2-bit index pairs
```

**Hardware execution (cuSPARSELt):**

The Sparse Tensor Core never explicitly reconstructs the dense matrix. It reads the compressed values + metadata and skips zero-multiplications directly during the GEMM operation. No decompression step — all in-kernel.

**PyTorch API:**
```python
from torch.sparse import to_sparse_semi_structured

# After masking weights to satisfy 2:4 constraint:
w_masked = weight * mask_24              # must already satisfy 2:4 pattern
w_sparse = to_sparse_semi_structured(w_masked)  # compressed format

# Forward pass automatically uses cuSPARSELt:
output = F.linear(input, w_sparse, bias)
```

---

#### 3. Mask Computation

The 2:4 mask is computed per-row, in groups of 4 along the input (column) dimension — keep the 2 highest-magnitude weights per group:

```python
def compute_24_mask(weight: Tensor) -> Tensor:
    # weight: [out_features, in_features]
    out, in_ = weight.shape
    w = weight.detach().view(out, -1, 4)      # [out, in/4, 4]
    _, keep_idx = w.abs().topk(2, dim=-1)     # top-2 per group of 4
    mask = torch.zeros_like(w).scatter_(-1, keep_idx, 1.0)
    return mask.view(out, in_)                # [out, in], binary
```

---

#### 4. Training with Frozen Mask (STE)

During Phase 4, the mask is **computed once** then **frozen**. Only the non-zero weight values adapt.

```python
class Sparse24Linear(nn.Linear):
    mask_24: Tensor   # frozen, registered as buffer (not parameter)

    def forward(self, x):
        # Forward: apply mask (sparse behavior)
        # Backward: gradient flows through ALL weights — Straight-Through Estimator
        return F.linear(x, self.weight * self.mask_24, self.bias)
```

**Why frozen mask?**  
Dynamic mask (re-evaluated each N steps) risks oscillation — channels repeatedly switching in/out. A short recovery phase (5–10k steps) with fixed structure is more stable. After training, the model satisfies the 2:4 constraint exactly and can be converted to `SparseSemiStructuredTensor`.

---

#### 5. Constraints

**A. Dimension alignment (cuSPARSELt fp16 requirement)**

Both row and column dimensions must be divisible by 64.

| Layer | Row dim | Col dim | Notes |
|---|---|---|---|
| `to_q` | inner_dim | residual channels | Both need alignment |
| `to_k` (cross) | inner_dim | 768 (text encoder) | 768 = 64×12 ✅ always aligned |
| `to_v` (cross) | inner_dim | 768 | Same ✅ |
| `to_out[0]` | residual channels | inner_dim | Both need alignment |

SD-Turbo deep blocks (head_dim=160): 6 heads → inner_dim=960 ✅. Exception: `down_blocks.2.attn1` after 1 head pruned → 7×160=1120 ✗.

**Resolution:** Re-run last pruning round with a `--align64` flag in `gen_pruning_config.py` that rounds target channel counts down to the nearest multiple of 64. Cost: at most 63 fewer channels per layer.

**B. Data type**

cuSPARSELt only supports `fp16`, `bf16`, `int8`. Student trains in fp32 — convert to fp16 before Phase 4 or apply sparsification post-hoc before inference conversion. Mask computation can be done in fp32.

**C. Contiguous memory**

`to_sparse_semi_structured` requires `weight.is_contiguous() == True`. After weight surgery (trimming/pruning), call `.contiguous()` on affected tensors.

**D. Minimum layer size**

Very small layers (inner_dim < 128) are unlikely to see real speedup from sparse kernels. In practice all target layers (mid/deep zone attention) are well above this threshold.

**E. PyTorch version**

`torch.sparse_semi_structured` requires PyTorch ≥ 2.1. `SparseSemiStructuredTensor` is stable from 2.2+.

**F. Mask re-application after optimizer step**

During STE training, the optimizer updates ALL weights (including currently-zero ones). After each step, the zero weights become slightly non-zero. Since mask is frozen, the `forward` still zeroes them out — forward behavior is correct. After training ends, apply `weight.data *= mask_24` once before converting to sparse format.

---

#### 6. Target Layers

**Attention Linear layers in mid-zone and deep-zone blocks:**
- `to_q`, `to_k`, `to_v`, `to_out[0]` in `mid_block`, `down_blocks.2/3`, `up_blocks.0/1`
- FFN (GEGLU) linear layers: optional, add if dims are aligned
- ResNet Conv2d: skip — PyTorch 2:4 conv support is limited

---

#### 7. New Files / Modified Files

| File | Change |
|---|---|
| `prune/sp_sparse24.py` | **New** — `Sparse24Linear`, `compute_24_mask()`, model replace/restore/convert utilities |
| `prune/gen_pruning_config.py` | Add `--align64` flag to round channel counts to multiples of 64 |
| `prune/sp_distill.py` | Add `--sparse-24` flag: wraps target layers with `Sparse24Linear` before Phase 4 distillation |
| `prune/sp_distill_config.yaml` | Add `sparse_24: false`, `sparse_24_layers: [...]` keys |

---

#### 8. Expected Combined Benefit

| Step | Compression | GEMM cost |
|---|---|---|
| Baseline (original) | 858M params | M × N × K |
| After channel pruning | 647M params | ~0.74× (smaller M, N) |
| After 2:4 sparsity | same param count | ~0.37× (2× sparse kernel) |
| Combined | — | **~0.37× vs original** |